# PolyAletheia - Training on Colab

Run this notebook on Google Colab to train using GPU.
This notebook clones the repository directly from GitHub.

In [ ]:
# 1. Setup & Clone
# Replace with your actual repo URL if different
REPO_URL = "https://github.com/ChandraguptSharma07/PolyAletheia.git"

import os
if not os.path.exists("PolyAletheia"):
    !git clone $REPO_URL
    %cd PolyAletheia
else:
    %cd PolyAletheia
    !git pull
    
# Install dependencies
!pip install -r requirements.txt

In [ ]:
# 2. Check GPU & Data
import torch
import wandb
import os

if torch.cuda.is_available():
    print(f"GPU available: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: Running on CPU. Enable GPU in Runtime > Change runtime type.")

# Check for data
if not os.path.exists("train.csv") and not os.path.exists("train_split.csv"):
    print("Data missing. Please upload 'train.csv' (dataset) or pre-split CSVs.")
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        pass

# Login to WandB
print("Logging into WandB... (optional)")
try:
    wandb.login()
except:
    print("WandB login failed or skipped.")

In [ ]:
# 3. Run Training

from train import train, validate, PolymerDataset, DataLoader, AdamW, get_linear_schedule_with_warmup, masked_mse_loss
from model import PolymerPredictor
from tokenizer import get_tokenizer

# Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = get_tokenizer()
model = PolymerPredictor().to(device)

print("Loading data...")
# Ensure data splits are present (repo should contain them or dvc pull)
if not os.path.exists("train_split.csv"):
    print("Generating splits...")
    # make sure data splits can find train.csv
    if os.path.exists("train.csv"):
        from data_splits import create_splits
        create_splits()
    else:
        raise FileNotFoundError("train.csv not found! Please upload it.")

train_ds = PolymerDataset("train_split.csv", tokenizer)
val_ds = PolymerDataset("val_split.csv", tokenizer)

train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=32)

optimizer = AdamW(model.parameters(), lr=1e-4)
epochs = 20
scheduler = get_linear_schedule_with_warmup(optimizer, 0, len(train_dl)*epochs)

# Init WandB
try:
    wandb.init(project="polyaletheia-colab", config={
        "model": "ChemBERTa",
        "epochs": 20,
        "batch_size": 32
    })
except Exception as e:
    print(f"WandB init failed: {e}\nRunning in disabled mode.")
    wandb.init(mode="disabled")

best_val_loss = float("inf")

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")
    train_loss = train(model, train_dl, optimizer, scheduler, device)
    val_loss = validate(model, val_dl, device)
    print(f"Train Loss {train_loss:.4f}, Val Loss {val_loss:.4f}")
    
    # only log if wandb run is active
    if wandb.run is not None:
        wandb.log({"train_loss": train_loss, "val_loss": val_loss})
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_model_colab.pth")
        print("Saved best model.")
        
wandb.finish()